In [ ]:
import pandas as pd 
import numpy as np
from sklearn.metrics import accuracy_score,recall_score,precision_score,average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone

In [ ]:
# 02 Modeling
# Input: `data/processed/modeling_table.csv, built at the end of notebook 01 (one row per labelable order, 96,470 rows).
# - Target: `is_late` (1 if the order arrived after the promised day).
# - Features: promised_days, n_items, n_sellers, total_freight, total_price, total_weight_g, dist_km_max, weight_missing. All are known at purchase time.
# - Missing values (distance 480 orders, weight 22 orders) are left as NaN here and imputed inside the pipeline.

In [ ]:
# read the modelling table
data = pd.read_csv("../data/processed/modelling_table.csv")

In [ ]:
data.shape

In [ ]:
data.dtypes

In [ ]:
data.isna().sum()

In [ ]:
# order_purchase_timestamp to datetime
data["order_purchase_timestamp"] = pd.to_datetime(data["order_purchase_timestamp"])

In [ ]:
data.dtypes

In [ ]:
# month column for splitting
data["month_year"]= data["order_purchase_timestamp"].dt.to_period('M')

In [ ]:
data.head()

In [ ]:
data.groupby("month_year").agg(
    count = ("order_id","count"),
    is_late = ("is_late","mean")
)

In [ ]:
# Time-based split
# The model is meant to predict future orders from past ones, so the split follows purchase date instead of shuffling rows.
# - Train: orders before 2018-06-01 (77,867 orders, about 7.5% late).
# - Test: orders from 2018-06-01 to the end of the data (18,603 orders, about 3.6% late).
# - The test set is locked. It is used once, at the end. Model selection and tuning use time-ordered cross-validation inside the training set.

# Note: the late rate differs between the two periods (monthly late rates range from about 1% to 19%), so precision on the test set will be lower than on training for any model. Compare models against the baseline, not against each other's raw numbers across periods.

In [ ]:
# training set from 2016 to may 2018
train_data = data[(data["month_year"] >= "2016-09") & (data["month_year"] <= "2018-05")]

In [ ]:
train_data.isna().sum()

In [ ]:
test_data = data[(data["month_year"] >= "2018-06") & (data["month_year"] <= "2018-08")]

In [ ]:
test_data.shape

In [ ]:
test_data.isna().sum()

In [ ]:
# separate the train inputs from the answer
X_train = train_data[["promised_days","n_items","n_sellers","total_freight","total_price","total_weight_g","dist","weight_missing"]]

In [ ]:
y_train = train_data["is_late"]

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
# separate the test inputs from the answer
X_test = test_data[["promised_days","n_items","n_sellers","total_freight","total_price","total_weight_g","dist","weight_missing"]]

In [ ]:
y_test = test_data["is_late"]

In [ ]:
X_test.shape[0] + X_train.shape[0]

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
X_test.isna().sum()

In [ ]:
X_train.isna().sum()

In [ ]:
# late rate mean for both train and test

In [ ]:
y_train.mean()

In [ ]:
y_test.mean()

In [ ]:
# left order_id and dates columns because a column goes into X only if its known at the purchase time, its meaningful for predicting lateness, and will still make sense for future orders.
# IDs and raw dates fails the last two tests
# The model will memorize things like "march 2018 was a bad month"
# The model will memorize an order id and know which specific orders were late 

In [ ]:
# The baseline
y_pred_baseline = np.zeros(len(y_train),dtype=int)

In [ ]:
y_pred_baseline

In [ ]:
accuracy_score = accuracy_score(y_train,y_pred_baseline)
accuracy_score

In [ ]:
recall = recall_score(y_train,y_pred_baseline)
recall

In [ ]:
precision = precision_score(y_train,y_pred_baseline,zero_division=0)
precision  

In [ ]:
# Baseline: "always on time"
# A model that predicts "on time" for every order.
# - Accuracy: about 92.5% (it is right whenever the order is on time).
# - Recall for late orders: 0 (it catches none).
# - Precision: undefined (it makes no late predictions).

# Accuracy is not used to judge models in this project, because it rewards ignoring the minority class. Models are judged by precision at a minimum recall (see decisions.md).

In [ ]:
# Model 1: logistic regression pipeline
# Pipeline steps, in order:
# 1. `SimpleImputer(strategy="median")`: fills missing distance (480 orders) and weight (22 orders).
# 2. `StandardScaler`: puts features on a common scale, because logistic regression is sensitive to scale (distance goes up to thousands of km, n_sellers is 1 or 2).
# 3. `LogisticRegression`.

# Everything that is learned from data (the medians, the means and standard deviations) sits inside the pipeline, so each fold learns it from its own training part only. This avoids leaking validation or test rows into preprocessing.
# The median stored in the fitted imputer equals the training median of dist computed by hand.

In [ ]:
pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy='median')),
    ("scaler",StandardScaler()),
    ("model",LogisticRegression())
])

In [ ]:
pipeline.fit(X_train,y_train)

In [ ]:
# inspect the imputer
pipeline.named_steps["imputer"].statistics_      

In [ ]:
pipeline.named_steps["imputer"].feature_names_in_

In [ ]:
X_train["dist"].median()

In [ ]:
# inspect the model
pipeline.named_steps["model"].coef_

In [ ]:
pipeline.named_steps["model"].intercept_

In [ ]:
pipeline.named_steps["model"].classes_

In [ ]:
pipeline.predict_proba(X_train)

In [ ]:
X_train.columns.tolist()

In [ ]:
# inspect scaler
pipeline.named_steps["scaler"].mean_

In [ ]:
X_train.mean()

In [ ]:
pipeline.named_steps['scaler'].scale_

In [ ]:
pipeline.named_steps['scaler'].var_

In [ ]:
pipeline.named_steps['scaler'].transform(
    pipeline.named_steps['imputer'].transform(X_train.iloc[[0]])
)

In [ ]:
# Coefficients 
# odds-comparing late orders against on-time orders
# probability-the proportion of late orders out of all orders
# p(late) = odds/(1+odds)
# log odds = Intercept + all feature contributions 


# each feature has its own original value, own mean and standard deviation, own standardized value and own learned coefficient

In [ ]:
train_data.head() 

In [ ]:
# sort values by purchase date
train_data_sorted = train_data.sort_values(by="order_purchase_timestamp")
train_data_sorted.head()

In [ ]:
# check whether the dates are in the ascending order
train_data_sorted["order_purchase_timestamp"].is_monotonic_increasing

In [ ]:
# sorted training data
X_train_sorted = train_data_sorted[["promised_days","n_items","n_sellers","total_freight","total_price","total_weight_g","dist","weight_missing"]]
X_train_sorted.head()

In [ ]:
y_train_sorted = train_data_sorted["is_late"]
y_train_sorted.head()

In [ ]:
y_train_sorted.shape

In [ ]:
# Time-ordered cross-validation
# - Training data is sorted by purchase date, with X and y kept aligned. `TimeSeriesSplit` cuts by row position, so unsorted data would let a fold train on the future.
# - 5 folds. Each fold trains on all earlier orders and validates on the next block of 12,977 orders.
# - Metric: average precision (AP), which summarizes the precision-recall trade-off across all thresholds and suits a rare positive class.
# - Accuracy is not used, because only about 7.5% of training orders are late.
# - A model with no skill scores an AP equal to the late rate of the validation block, so each fold's AP is compared with that fold's own late rate (the ratio AP / late rate).

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)  

for train_idx,val_idx in tscv.split(X_train_sorted):
    print(f"Train: {train_idx}, val: {val_idx}")

In [ ]:
for train_idx,val_idx in tscv.split(X_train_sorted):
    X_fold_train = X_train_sorted.iloc[train_idx]
    X_fold_val = X_train_sorted.iloc[val_idx]

    y_fold_train = y_train_sorted.iloc[train_idx]
    y_fold_val = y_train_sorted.iloc[val_idx]

    fold_pipeline = clone(pipeline)

    fold_pipeline.fit(X_fold_train,y_fold_train)

    y_val_proba = fold_pipeline.predict_proba(X_fold_val)[:,1]  

    ap = average_precision_score(y_fold_val,y_val_proba)

    late_rate = y_fold_val.mean()

    len(X_fold_train)
    len(X_fold_val)

    print(f"Train:{len(X_fold_train)}",f"Validation: {len(X_fold_val)}",f"Late rate: {late_rate:.3f}",f"Average Precision: {ap:.3f}")


In [ ]:
# The ratio is better because it measures AP relative to each fold's late rate, accounting for diffrences in how common late rate orders are across periods. This makes comparisons fairer than raw AP, which is strongly influenced by the late-order prevalence
# Mean ratio: 1.71.
# Interpretation:
# - The model beats the no-skill level in all five folds, so it has real but modest skill (1.3 to 2.5 times chance).
# - Raw AP varies mostly because the late rate varies across periods (3.4% to 17.4%), so the ratio is the fairer comparison.
# - Fold 4 has the highest late rate, probably the months around November 2017 to March 2018. The model's edge there is small relative to its AP.
# - Fold 1 trains on only about 13,000 early orders from a much smaller platform.
# - Consistent with the feature analysis, every feature is a weak signal on its own, so a modest improvement over chance is expected.

# The mean ratio of 1.71 is the number the next model must beat.